# Portfolio Manager: Analyse, Optimierung, Backtest

Python rechnet alles (Kurse in EUR, Risiko, Stresstests, Min-Variance / Max-Sharpe / HRP, Walk-forward-Backtest), der Agent interpretiert und schreibt den Report. Benchmark: MSCI ACWI (ACWI-ETF, in EUR umgerechnet), dazu FTSE All-World (VWCE.DE).

Eigenes Depot: CSV mit `ticker` und `weight` (oder `shares`) in `ai_in_finance/portfolios/` ablegen und in Zelle 3 eintragen. Die Datei bleibt auf deinem Laptop. Nur fuer Lehrzwecke, keine Anlageberatung.

**Echte Performance (optional):** Transaktions-Exporte von Scalable Capital, Trade Republic oder Smartbroker+ (CSV) in `ai_in_finance/portfolios/` legen und unten in `TRANSACTION_FILES` eintragen. Dann berechnet Python TTWROR, MWR/IRR, den MSCI ACWI mit denselben Zahlungen, Gewinn/Verlust je Position und die Wirkung jedes Kaufs/Verkaufs; der aktuelle Bestand kommt dann aus den Transaktionen. ISINs werden automatisch in Yahoo-Ticker uebersetzt (`portfolios/isin_map.csv`, von Hand korrigierbar). Die Exporte bleiben lokal (gitignored).

In [ ]:
# 1) Pfade und Imports
import os, sys, json
from pathlib import Path

ROOT = Path.cwd().resolve().parents[1]      # ...\FinRobot
AIF = ROOT / "ai_in_finance"
FR = ROOT / "finrobot_autogen"
sys.path.insert(0, str(AIF))

import autogen
from finrobot.utils import register_keys_from_json
from lib import portfolio_engine as pe
from lib.portfolio_manager import run_portfolio_review
from lib import performance as pf
from lib.portfolio_manager import load_performance


In [ ]:
# 2) Modell und Schluessel
MODEL = "anthropic/claude-haiku-5.5"

base = json.load(open(FR / "OAI_CONFIG_LIST", encoding="utf-8"))
entry = next((e for e in base if e["model"] == MODEL), None) or dict(base[0], model=MODEL)
llm_config = {"config_list": [entry], "timeout": 300, "temperature": 0.3,
              "parallel_tool_calls": False, "max_tokens": 12000}
if MODEL.startswith("google/"):
    llm_config["extra_body"] = {"reasoning": {"effort": "low"}}
register_keys_from_json(str(FR / "config_api_keys"))


In [ ]:
# 3) Portfolio und Einstellungen; Analyse + Backtest berechnen (ohne LLM, ca. 1-2 Minuten)
# Variante A: nur Bestand (ticker + weight oder shares). Variante B: Transaktions-Exporte -> echte Performance.
PORTFOLIO_FILE = AIF / "portfolios" / "sample_portfolio.csv"
TRANSACTION_FILES = [AIF / "portfolios" / "sample_transactions.csv"]   # Musterdepot mit Historie; [] = nur Bestand aus PORTFOLIO_FILE
HOLDINGS_FILES = []        # nur Positions-Exporte ohne Kaufhistorie (z.B. "Depotpositionen"); Aktivitaeten-Exporte gehoeren in TRANSACTION_FILES
# z.B. [AIF / "portfolios" / "scalable.csv", AIF / "portfolios" / "traderepublic.csv", AIF / "portfolios" / "smartbroker.csv"]
NAME, TITLE = "sample", "Sample Portfolio"
RF, CAP, SECTOR_CAP = 0.02, 0.10, 0.35      # risikofreier Zins (EUR), max. Gewicht je Titel / je Sektor

performance = None
if TRANSACTION_FILES:
    performance, tx, issues = load_performance([str(f) for f in TRANSACTION_FILES])
    print(pf.performance_text(performance)); print()
    print("Hinweise:", *performance["warnings"], sep="\n- ")
    portfolio = pf.holdings_portfolio(performance)          # aktueller Bestand aus den Transaktionen
elif not HOLDINGS_FILES:
    portfolio = pe.load_portfolio(PORTFOLIO_FILE)
else:
    portfolio = []
if HOLDINGS_FILES:                                          # Positionen ohne Kaufhistorie dazu (nicht in der Performance)
    from lib import transactions as tr
    extra, iss = tr.read_all_holdings([str(f) for f in HOLDINGS_FILES])
    print(f"{len(extra)} Positionen aus Positions-Exporten", *iss, sep="\n- ")
    portfolio = tr.merge_positions(portfolio, extra)
analysis = pe.analyze(portfolio, rf=RF, cap=CAP, sector_cap=SECTOR_CAP)
sectors = [analysis["fund"][t]["sector"] for t in analysis["tickers"]]
backtest = pe.backtest(analysis["prices"], analysis["tickers"], analysis["weights"][analysis["tickers"]].values,
                       rf=RF, cap=CAP, sectors=sectors, sector_cap=SECTOR_CAP)
print(pe.analysis_text(analysis)); print(); print(pe.backtest_text(backtest))

In [ ]:
# 4) Agent schreibt den Report
save_dir = f"../reports/portfolio_{NAME}_{MODEL.split('/')[-1]}"
run_portfolio_review(llm_config, None if (TRANSACTION_FILES or HOLDINGS_FILES) else str(PORTFOLIO_FILE), save_dir, name=NAME, title=TITLE,
                     rf=RF, cap=CAP, sector_cap=SECTOR_CAP, analysis=analysis, backtest=backtest,
                     performance=performance, transaction_files=[str(f) for f in TRANSACTION_FILES],
                     holdings_files=[str(f) for f in HOLDINGS_FILES])

In [ ]:
# 5) PDF anzeigen
import io, glob
import pymupdf as fitz
from PIL import Image
pdf = glob.glob(f"{save_dir}/*_portfolio_review.pdf")
for page in (fitz.open(pdf[0]) if pdf else []):
    display(Image.open(io.BytesIO(page.get_pixmap(dpi=100).tobytes("png"))))
